In [1]:
# HCP YA data split


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold

df = pd.read_csv('/home/jovyan/nh26_td_gamb_diff_FC_best/data/HCP_YA_subjects_2026_07_20_14_29_24.csv')
diffusion_sub = pd.read_csv('/home/jovyan/nh26_td_gamb_diff_FC_best/data/diffusion_subjects.csv')
fmri_sub = pd.read_csv('/home/jovyan/nh26_td_gamb_diff_FC_best/data/with_gambling.csv')
dd_sub = pd.read_csv('/home/jovyan/nh26_td_gamb_diff_FC_best/data/DD_AUC_targets.csv')

# # Filter to subjects with full 3T completion
df_filtered = df[df['3T_Full_MR_Compl'] == True].copy()
print(f"Original N: {len(df)}")
print(f"Filtered N (3T complete): {len(df_filtered)}")

diffusion_ids = set(diffusion_sub['subjectID'])
fmri_ids = set(fmri_sub['0'])
dd_ids = set(dd_sub['Subject'][dd_sub['DDisc_AUC_40K'].notna()])

df_final = df_filtered[
    df_filtered['Subject'].isin(diffusion_ids) & df_filtered['Subject'].isin(fmri_ids) 
    & df_filtered['Subject'].isin(dd_ids)
    ].copy()

print(f"3T complete N: {len(df_filtered)}")
print(f"Diffusion available N: {len(diffusion_ids)}")
print(f"fMRI/gambling available N: {len(fmri_ids)}")
print(f"DD behavioral measures available N: {len(dd_ids)}")

print(f"Final N (all three criteria met): {len(df_final)}")

Original N: 1206
Filtered N (3T complete): 889
3T complete N: 889
Diffusion available N: 1041
fMRI/gambling available N: 1084
DD behavioral measures available N: 887
Final N (all three criteria met): 864


In [3]:

df_final['age_group'] = df_final['Age'].astype(str).str.strip()
df_final['strata'] = df_final['age_group'] + '_' + df_final['Gender'].astype(str)

print(df_final['strata'].value_counts())
print(f"\nNumber of unique strata: {df_final['strata'].nunique()}")

strata
26-30_F    201
31-35_F    176
26-30_M    175
22-25_M    123
31-35_M    109
22-25_F     71
36+_F        6
36+_M        3
Name: count, dtype: int64

Number of unique strata: 8


In [4]:
train_df, test_df = train_test_split(
    df_final,
    test_size=0.2,
    stratify=df_final['strata'],
    random_state=42
)

print(f"Train N: {len(train_df)}, Test N: {len(test_df)}")
print("\nTrain strata counts:")
print(train_df['strata'].value_counts())
print("\nTest strata counts:")
print(test_df['strata'].value_counts())

Train N: 691, Test N: 173

Train strata counts:
strata
26-30_F    161
31-35_F    141
26-30_M    140
22-25_M     98
31-35_M     87
22-25_F     57
36+_F        5
36+_M        2
Name: count, dtype: int64

Test strata counts:
strata
26-30_F    40
31-35_F    35
26-30_M    35
22-25_M    25
31-35_M    22
22-25_F    14
36+_M       1
36+_F       1
Name: count, dtype: int64


In [5]:
import matplotlib.pyplot as plt

train_prop = train_df['strata'].value_counts(normalize=True).sort_index()
test_prop = test_df['strata'].value_counts(normalize=True).sort_index()

comparison = pd.DataFrame({'train': train_prop, 'test': test_prop})
print(comparison)

            train      test
strata                     
22-25_F  0.082489  0.080925
22-25_M  0.141823  0.144509
26-30_F  0.232996  0.231214
26-30_M  0.202605  0.202312
31-35_F  0.204052  0.202312
31-35_M  0.125904  0.127168
36+_F    0.007236  0.005780
36+_M    0.002894  0.005780


In [6]:
print(train_df)

      Subject Release Acquisition Gender    Age  3T_Full_MR_Compl  \
830    519647   S1200         Q13      M  26-30              True   
687    308129    S900         Q11      M  22-25              True   
1093   837560    S500         Q05      M  26-30              True   
835    523032    S900         Q09      M  31-35              True   
541    200614      Q1         Q01      F  31-35              True   
...       ...     ...         ...    ...    ...               ...   
896    580650    S900         Q09      F  31-35              True   
140    125525      Q1         Q01      F  31-35              True   
564    205725      Q2         Q02      F  22-25              True   
504    194746    S900         Q07      M  31-35              True   
233    141826    S500         Q04      F  26-30              True   

      7T_Full_MR_Compl  MEG_FullProt_Compl age_group   strata  
830              False               False     26-30  26-30_M  
687              False               False 

In [7]:
# Reset index for clean row numbers
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Save training set
train_df.to_csv('/home/jovyan/nh26_td_gamb_diff_FC_best/data/train_split.csv', index=False)

# Save test set
test_df.to_csv('/home/jovyan/nh26_td_gamb_diff_FC_best/data/test_split.csv', index=False)

print(f"Train saved: {len(train_df)} subjects")
print(f"Test saved: {len(test_df)} subjects")

Train saved: 691 subjects
Test saved: 173 subjects
